In [1]:
%env HUGGINGFACE_HUB_CACHE= models
%env TORCH_HOME=models

env: HUGGINGFACE_HUB_CACHE=models
env: TORCH_HOME=models


In [2]:
import pandas as pd
import torch
import io
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [3]:
df = pd.read_csv('data/employee.csv')

buffer = io.StringIO()
df.info(buf=buffer)
df_structure = buffer.getvalue()

print("--- Captured Schema ---")
print(df_structure)

--- Captured Schema ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   empID      20 non-null     object
 1   empName    20 non-null     object
 2   empDept    20 non-null     object
 3   empSalary  20 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 772.0+ bytes



In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
model_name = "microsoft/phi-2"

model_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    # device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
llm_pipeline = pipeline(
    "text-generation",
    model=model_llm,
    tokenizer=tokenizer,
    device=device,
    max_length= 500
)

Device set to use cuda


In [7]:
user_query = input("Enter your query: ")

In [ ]:
prompt = f"""
You are an expert Python assistant. You translate natural language questions into a single, executable line of Python code for a pandas DataFrame named `df`.

DataFrame Schema:
{df_structure}

Examples:

Question: "How many employees are there?"
Code: print(len(df))

Question: "Show me the top 3 highest paid employees."
Code: print(df.sort_values('empSalary', ascending=False).head(3))

Question: "What is the total salary of the 'Sales' department?"
Code: print(df[df['empDept'] == 'Sales']['empSalary'].sum())

Question: "Give me all employees in 'Engineering'."
Code: print(df[df['empDept'] == 'Engineering'])

Current Task:

Question: "{user_query}"
Code:"""

print("--- Sending Improved Prompt to Phi-2 ---")
print(prompt) # You can uncomment this to debug the full prompt

torch.manual_seed(42)
sequences = llm_pipeline(
    prompt,
    do_sample=False, # Set to False for deterministic code generation
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

print("\n--- LLM Response (Raw) ---")
generated_text = sequences[0]['generated_text']
print(generated_text)

# --- This is the new, corrected Parsing Logic ---
try:
    # 1. Remove the prompt you sent
    response_only = generated_text.replace(prompt, "").strip()

    # 2. Get the first line of code
    generated_code = response_only.split('\n')[0].strip()

    print(f"\n--- Extracted Code ---")
    print(generated_code)

    # --- Step 4: Execute the Code (With a Basic Safety Check) ---
    print(f"\n--- Running Code ---")

    # !! WARNING: Never use eval() in production.
    # This is a simple safety check for this demo.
    # A real app must use a secure sandbox.

    allowed_terms = ['df', 'print', 'len', 'sort_values', 'head', 'sum', 'mean', 'count', 'shape', 'empID', 'empName', 'empDept', 'empSalary']

    if "import" in generated_code or "os." in generated_code:
        print(f"Execution Aborted: Generated code contains unsafe commands.")
    else:
        # We assume the code is safe for this demo
        exec(generated_code)

except Exception as e:
    print(f"\n--- Error ---")
    print(f"Failed to parse or run code: {e}")

--- Sending Improved Prompt to Phi-2 ---

You are an expert Python assistant. You translate natural language questions into a single, executable line of Python code for a pandas DataFrame named `df`.

DataFrame Schema:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   empID      20 non-null     object
 1   empName    20 non-null     object
 2   empDept    20 non-null     object
 3   empSalary  20 non-null     int64 
dtypes: int64(1), object(3)
memory usage: 772.0+ bytes


Examples:

Question: "How many employees are there?"
Code: print(len(df))

Question: "Show me the top 3 highest paid employees."
Code: print(df.sort_values('empSalary', ascending=False).head(3))

Question: "What is the total salary of the 'Sales' department?"
Code: print(df[df['empDept'] == 'Sales']['empSalary'].sum())

Question: "Give me all employees in 'Engineering'."
Code: print(df

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:

import torch

# --- Get User Input ---
user_query = input("Enter your query: ")

# ======================================================================
#  ROLE 1: Translate Natural Language to PANDAS CODE
# ======================================================================

# --- Prompt 1: The "Text-to-Code" Few-Shot Prompt ---
# NOTE: The examples are changed to NOT use print().
# This makes the output a value we can 'eval()' and capture.

prompt_role_1 = f"""
You are an expert Python assistant. You translate natural language questions into a
single, executable line of Python code for a pandas DataFrame named `df`.
Do not include any explanation, just the code.

DataFrame Schema:
{df_structure}

Examples:

Question: "How many employees are there?"
Code: len(df)

Question: "Show me the top 3 highest paid employees."
Code: df.sort_values('empSalary', ascending=False).head(3)

Question: "What is the total salary of the 'Sales' department?"
Code: df[df['empDept'] == 'Sales']['empSalary'].sum()

Question: "Give me all employees in 'Engineering'."
Code: df[df['empDept'] == 'Engineering']

Current Task:

Question: "{user_query}"
Code:"""

print("--- [Role 1] Sending Text-to-Code Prompt ---")
torch.manual_seed(42)
sequences = llm_pipeline(
    prompt_role_1,
    do_sample=False,
    num_return_sequences=1,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=60 # Make sure this is still set from Cell 6
)

# --- Parse and Execute Code ---
generated_code = ""
data_result = None

try:
    # 1. Remove the prompt you sent
    response_only = sequences[0]['generated_text'].replace(prompt_role_1, "").strip()

    # 2. Get the first line of code
    generated_code = response_only.split('\n')[0].strip()

    print(f"   > Extracted Code: {generated_code}")

    # 3. !! WARNING: 'eval()' is unsafe in production. !!
    # This is for demo purposes. Use a secure sandbox.
    data_result = eval(generated_code)

    print(f"   > Captured Data: {data_result}")

except Exception as e:
    print(f"\n--- [Role 1] Error ---")
    print(f"Failed to parse or run code: {e}")
    data_result = "Error: Could not retrieve data."


# ======================================================================
#  ROLE 2: Translate PANDAS DATA to Natural Language
# ======================================================================

if data_result is not None:
    # --- Prompt 2: The "Data-to-Text" (Guidance) Prompt ---
    # This prompt teaches the model to be a friendly interpreter.

    prompt_role_2 = f"""
    You are a helpful assistant. Given a user's question and the raw data
    result, provide a friendly, human-readable answer.

    Question: "How many employees are there?"
    Data Result: 20
    Answer: There are 20 employees in total.

    Question: "What is the total salary of the 'Sales' department?"
    Data Result: 95000
    Answer: The total salary for the Sales department is 95,000.

    Question: "Show me the top 3 highest paid employees."
    Data Result:
         empID   empName  empDept  empSalary
    10  E11    Karan    Sales      65000
    1    E02    Jane     Dev        60000
    15  E16    Ravi     Dev        60000
    Answer: Here are the top 3 highest paid employees:
    Karan (Sales, 65000), Jane (Dev, 60000), and Ravi (Dev, 60000).

    Current Task:

    Question: "{user_query}"
    Data Result: {data_result}
    Answer:"""

    print("\n--- [Role 2] Sending Data-to-Text Prompt ---")
    torch.manual_seed(42)
    sequences_2 = llm_pipeline(
        prompt_role_2,
        do_sample=False,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=100 # Give it more space for a friendly answer
    )

    try:
        # --- Parse and Display the Final Answer ---
        response_text_2 = sequences_2[0]['generated_text'].replace(prompt_role_2, "").strip()
        friendly_answer = response_text_2.split('\n')[0].strip()

        print("\n====================================")
        print(f"Final Answer: {friendly_answer}")
        print("====================================")

    except Exception as e:
        print(f"\n--- [Role 2] Error ---")
        print(f"Failed to parse friendly answer: {e}")